In [1]:
# Import 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Document load
pdf_url = "https://www.adobe.com/be_en/active-use/pdf/Alice_in_Wonderland.pdf"
loader = PyPDFLoader(pdf_url)
pages = loader.load()

In [4]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len 
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Split into 245 chunks


In [7]:
# Creating Embeddings 
embeddings = OllamaEmbeddings(model = "nomic-embed-text")
vector_store = LanceDB.from_documents(chunks , embedding=embeddings)
print(f"Created vector store with {len(chunks)} documents")

Created vector store with 245 documents


In [9]:
# Rag Query 
llm = ChatOllama(model = "llama3:latest" , temperature= 0)
retriever = vector_store.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

rag_chain = (
    {
        "context" : retriever | format_docs,
        "question" : RunnablePassthrough()
    } 
    | prompt | llm | StrOutputParser()
)

In [10]:
# Ask questions about Alice in Wonderland
questions = [
    "Who is the main character and what happens at the beginning of the story?",
    "What did the Caterpillar ask Alice?",
    "Describe the Mad Hatter's tea party.",
    "What happened at the trial at the end of the story?"
]

for q in questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

Q: Who is the main character and what happens at the beginning of the story?
A: The main character is Alice. At the beginning of the story, Alice has just run out of her house and found a crowd of little animals and birds waiting outside. She sees a poor little lizard named Bill being held up by two guinea-pigs who are giving him something from a bottle. The creatures rush at Alice, but she runs off as hard as she can.
------------------------------------------------------------
Q: What did the Caterpillar ask Alice?
A: The Caterpillar asked Alice "Who are you?"
------------------------------------------------------------
Q: Describe the Mad Hatter's tea party.
A: Based on the provided context, here is a description of the Mad Hatter's tea party:

The Mad Hatter's tea party is a bizarre and chaotic gathering where Alice finds herself seated at a large table with the March Hare and the Dormouse. The table is laid out with only tea, no wine or other refreshments are present. The March Ha